### Goal Objective:
    Goal: Know exactly what data you have, what you don't have, and what you cannot use.



**LEVEL 1: Data Understanding**
Notebook `01_data_understanding.ipynb`. No ML. Check shape, dtypes, duplicates, missing values, target distribution. Answer the questions: how many applications? how much fraud? what products/cities? what features are categorical/numerical? where are the missing values? where is potential leakage?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl

### 01 – Dataset Exploration →
 ### Information about the dataset


In [3]:
df = pd.read_excel('../data/raw/sample.xlsx')
df.head()

,Дата заявки,Код заявки,Дата УДЛ клиента,Кредитный продукт,Заявленный официальный доход с анкеты,Менеджер,Запрошенная сумма кредита,Город,БИН работодаетя с отчета с ГЦВП,Просрочки за последние 6 месяцев в днях с кредитной истории ПКБ,Кол-во завершенных займов с кредитной истории ПКБ,Сумма остатка по всем займам с кредитной истории ПКБ,Кол-ао действующих займов клиента кредитной истории ПКБ,Кол-во запросов в ПКБ за последние 30 дней,Срок займа в мес,Признак вызревания CENZ_15_1,FPD_15,Признак вызревания CENZ_15_2,SPD_15,Активная просрочка в днях
0,2022-06-24,KAR.00095934,2006-12-14,Автокредитов,308260.0,Налыбек,2800000,Караганд,NaN,0.0,2.0,2290527.0,3,77.0,NaN,1,0,1,0,0
1,2022-06-23,1003534418,2013-01-10,Кредит налич,270000.0,Уштенова,1500000,Нур-Султ,990240003263/,0.0,9.0,2016655.0,4,36.0,5.0,1,0,1,0,0
2,2022-05-26,BCC_5407,2017-09-20,#картакарта_v,180000.0,Пользователь,70000,Тараз,991040000928/,9.0,4.0,347444.0,2,27.0,11.0,0,0,0,0,0
3,2022-05-15,BCC_1976,2006-08-28,#картакарта_v,800000.0,Пользователь,200000,Алматы,981141001567/,0.0,9.0,6297948.0,10,26.0,6.0,0,0,0,0,0
4,2022-04-29,ALM.00285810,2021-02-19,Автокредитов,600000.0,Мешітбай,11500000,Алматы,890111301342/,2.0,7.0,1974935.0,4,25.0,3.0,1,0,1,0,39


In [4]:
df.shape

(25536, 20)

### Rename the columns to facilitate a more nuanced understanding of the dataset.

In [5]:
rename_map = {
    'Дата заявки': 'application_date',
    'Код заявки': 'application_id',
    'Дата УДЛ клиента': 'client_id_document_date',
    'Кредитный продукт': 'credit_product',
    'Заявленный официальный доход с анкеты': 'declared_official_income',
    'Менеджер': 'manager',
    'Запрошенная сумма кредита': 'requested_loan_amount',
    'Город': 'city',
    'БИН работодаетя с отчета с ГЦВП': 'employer_bin_gcvp',
    'Просрочки за последние 6 месяцев в днях с кредитной истории ПКБ': 'delinquency_days_6m_credit_bureau',
    'Кол-во завершенных займов с кредитной истории ПКБ': 'completed_loans_count_credit_bureau',
    'Сумма остатка по всем займам с кредитной истории ПКБ': 'total_outstanding_balance_credit_bureau',
    'Кол-ао действующих займов клиента кредитной истории ПКБ': 'active_loans_count_credit_bureau',
    'Кол-во запросов в ПКБ за последние 30 дней': 'credit_bureau_inquiries_30d',
    'Срок займа в мес': 'loan_term_months',
    'Признак вызревания CENZ_15_1': 'cenz_15_1_maturity_flag',
    'FPD_15': 'fpd_15',
    'Признак вызревания CENZ_15_2': 'cenz_15_2_maturity_flag',
    'SPD_15': 'spd_15',
    'Активная просрочка в днях': 'active_delinquency_days',
}

df = df.rename(columns=rename_map)

Welcome back to the Royal Court, Treasurer! You have unrolled the Ancient Scroll (your DataFrame) and found 20 mystical columns. As your Master of Whispers, I will translate each rune for you. 

In the Kingdom of Finance, every Knight (borrower) must present a dossier. Some of these scrolls are useful for predicting if the Knight will steal the gold (Fraud/Default), some are just identification marks, and some are **prophecies of the future** that you must never use in your divination spell (because that would be cheating).

Here is the complete breakdown of your 20 columns, divided into four magical categories.

---

### 🛡️ Category 1: The Identity Scrolls (IDs and Dates)
*These scrolls tell you who the knight is and when they arrived. You usually don't feed raw IDs to the crystal ball, but you can extract powerful magic from dates.*

*   **`application_id` (The Quest Scroll Number):** 
    *   *What it is:* A unique number for the application.
    *   *Do you need it?* **No.** You must drop this before training your model. It is like a knight's name—it has no predictive power. If you leave it in, the crystal ball might memorize "Knight #892 is a thief" instead of learning *why* they are a thief.
*   **`application_date` (The Day the Knight Arrived):**
    *   *What it is:* The exact date the loan was requested.
    *   *Do you need it?* **Yes, but extract its magic.** Don't use the raw date. Extract the **Month**, **Day of the Week**, or **Is_Holiday**. Thieves (fraud rings) often strike on weekends or holidays when the Royal Guards (bank employees) are sleepy and less vigilant.
*   **`client_id_document_date` (The Knight's Birth Scroll):**
    *   *What it is:* The date the client's ID document was issued (or their birth date).
    *   *Do you need it?* **Yes, extract Age.** A knight who is 18 years old asking for 1,000,000 gold is highly suspicious. Also, notice this column has missing values (25,275 out of 25,536). *Tip:* Missing ID dates might mean the knight is hiding something. Create a new column `is_id_missing` and set it to 1 for these knights.

---

### 🏰 Category 2: The Knight's Manifest (Demographics & Loan Details)
*These are the claims the knight makes about themselves. Some are trustworthy; some are lies.*

*   **`credit_product` (The Type of Quest):**
    *   *What it is:* The type of loan (e.g., "Dragon Slaying Loan", "Castle Repair Loan").
    *   *Do you need it?* **Yes.** Certain quests are inherently riskier. A "Get Rich Quick" loan might have a 30% fraud rate, while a "Buy a Cow" loan might have a 2% fraud rate.
*   **`manager` (The Gatekeeper):**
    *   *What it is:* The bank employee who processed the application.
    *   *Do you need it?* **Yes.** Some gatekeepers are corrupt or careless. Fraud rings often bribe specific managers. You can calculate the historical fraud rate per manager.
*   **`city` (The Kingdom of Origin):**
    *   *What it is:* The city where the knight lives.
    *   *Do you need it?* **Yes.** Fraud is often geographically clustered. If a city is known for harboring bandits, applications from that city are riskier.
*   **`employer_bin_gcvp` (The Knight's Lord):**
    *   *What it is:* The employer's identification number.
    *   *Do you need it?* **Yes, but carefully.** This is a categorical column with many unique values (high cardinality). You can use it to create a feature: **"Is this a known, stable Lord, or a fake Lord?"** (Shell company). *Note:* It has missing values (21,789 out of 25,536). Missing employer = high risk.
*   **`declared_official_income` (Claimed Gold Earnings):**
    *   *What it is:* How much the knight *says* they earn.
    *   *Do you need it?* **Yes.** But never trust it alone. You must compare it to the requested loan amount to create a **Debt-to-Income (DTI) ratio**. "You claim to earn 100 gold a month, but you want a 10,000 gold loan? Guards, arrest this knight!"
*   **`requested_loan_amount` (The Gold They Desire):**
    *   *What it is:* The amount of gold requested.
    *   *Do you need it?* **Yes.** Greedy requests are a red flag. Also useful for calculating the DTI ratio.
*   **`loan_term_months` (The Length of the Quest):**
    *   *What it is:* How many months the knight has to repay.
    *   *Do you need it?* **Yes.** Longer quests have more time for the knight to encounter a dragon (lose their job, get sick) and default.

---

### 🔮 Category 3: The Crystal Ball (Credit Bureau History)
*This is the most powerful magic in the kingdom. The Credit Bureau is a spy network that tracks every knight's past behavior across all kingdoms. You absolutely need these.*

*   **`delinquency_days_6m_credit_bureau` (Days of Shame in the Last 6 Moons):**
    *   *What it is:* How many days late they were on other loans recently.
    *   *Do you need it?* **Yes.** This is a top predictor. If they are late to other lords, they will be late to you.
*   **`completed_loans_count_credit_bureau` (Successful Quests):**
    *   *What it is:* How many loans they paid back fully.
    *   *Do you need it?* **Yes.** A knight with 10 successful quests is trustworthy. A knight with 0 is an unknown risk.
*   **`total_outstanding_balance_credit_bureau` (Current Total Debt):**
    *   *What it is:* How much gold they currently owe to *other* lords.
    *   *Do you need it?* **Yes.** If they already owe 50,000 gold, why are they asking you for more? This indicates desperation.
*   **`active_loans_count_credit_bureau` (Current Active Quests):**
    *   *What it is:* How many loans they are currently paying off.
    *   *Do you need it?* **Yes.** If a knight has 15 active loans, they are over-leveraged. One small mishap and they will collapse.
*   **`credit_bureau_inquiries_30d` (Castles Visited Recently):**
    *   *What it is:* How many other banks pulled their credit report in the last 30 days.
    *   *Do you need it?* **Yes.** This is the "Desperation Meter." If they visited 10 castles in 10 days, they are desperately looking for gold. Desperate knights steal.

---

### ⚔️ Category 4: The Curses (The Outcomes / Target Variables)
*These columns tell you what happened **after** the gold was given. They are the "answers" to the exam. You must use them to train your crystal ball (model), but you must NEVER feed them into the crystal ball as features.*

*   **`fpd_15` (First Payment Default - The Immediate Theft):**
    *   *What it is:* A flag (0 or 1). Did the knight fail to make the very first payment within 15 days?
    *   *Do you need it?* **This is your main Target Variable.** If `fpd_15 = 1`, the knight took the gold and vanished. This is the closest thing to "Fraud" in your dataset.
*   **`spd_15` (Second Payment Default - The Serial Thief):**
    *   *What it is:* Did they default on the second payment within 15 days?
    *   *Do you need it?* **Yes, as a target or for analysis.** It tells you about the depth of the curse.
*   **`active_delinquency_days` (Currently Hiding from Guards):**
    *   *What it is:* How many days late they are *right now*.
    *   *Do you need it?* **Careful!** This is a consequence of the loan. If you are trying to predict fraud *at the moment of application*, you cannot use this, because at application time, this value doesn't exist yet. This is called **Data Leakage**.
*   **`cenz_15_1_maturity_flag` & `cenz_15_2_maturity_flag` (The Censored Fate Flags):**
    *   *What it is:* "Cenz" stands for Censored. In survival analysis, a loan is "censored" if it hasn't reached maturity yet, or if the observation window ended. These flags track the lifecycle of the loan.
    *   *Do you need it?* **For feature engineering only.** You can use them to understand the loan's timeline, but you must be very careful. They are closely tied to the target. If a loan is censored (flag=1), it means it hasn't defaulted *yet*. You usually drop these during the modeling phase unless you are doing advanced survival analysis.

---

### 🧙‍♂️ The Grand Summary: How to use this knowledge?

When you build your Antifraud Scoring model, your goal is to predict **`fpd_15`** (The First Payment Default) using **Categories 1, 2, and 3**.

**Your Magic Recipe (Feature Engineering):**
1.  **Drop:** `application_id` (useless).
2.  **Convert:** `application_date` -> Day of week / Month. `client_id_document_date` -> Age.
3.  **Encode:** `credit_product`, `city`, `manager` -> Convert to numbers (One-Hot Encoding or Target Encoding).
4.  **Create Ratios:** `requested_loan_amount` / `declared_official_income` = **DTI (Debt-to-Income)**. This is your most powerful spell.
5.  **Handle Missing:** Create flags for missing `employer_bin_gcvp` and `loan_term_months`. Missing data is often a red flag for fraud.
6.  **Train:** Use the crystal ball (XGBoost or Random Forest) to predict `fpd_15`.

Now, Treasurer, you have the map to the kingdom's gold. Go forth and build your defenses!

In [6]:
df.columns

Index(['application_date', 'application_id', 'client_id_document_date',
       'credit_product', 'declared_official_income', 'manager',
       'requested_loan_amount', 'city', 'employer_bin_gcvp',
       'delinquency_days_6m_credit_bureau',
       'completed_loans_count_credit_bureau',
       'total_outstanding_balance_credit_bureau',
       'active_loans_count_credit_bureau', 'credit_bureau_inquiries_30d',
       'loan_term_months', 'cenz_15_1_maturity_flag', 'fpd_15',
       'cenz_15_2_maturity_flag', 'spd_15', 'active_delinquency_days'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25536 entries, 0 to 25535
Data columns (total 20 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   application_date                         25536 non-null  datetime64[ns]
 1   application_id                           25536 non-null  object        
 2   client_id_document_date                  25275 non-null  datetime64[ns]
 3   credit_product                           25536 non-null  object        
 4   declared_official_income                 25536 non-null  float64       
 5   manager                                  25536 non-null  object        
 6   requested_loan_amount                    25536 non-null  int64         
 7   city                                     25536 non-null  object        
 8   employer_bin_gcvp                        21789 non-null  object        
 9   delinquency_days_6m_credit_bureau      

In [8]:
total_applications = df.shape[0]
print(f'Total application:{total_applications}')

Total application:25536


In [9]:
df['application_id'].unique().shape[0] 

25535

In [10]:
### Drop application_id -> unnecessary 
df = df.drop(columns=['application_id'])

In [11]:
### Missing Values
print("Missing Value Percentages:")
print(df.isna().mean() * 100)

Missing Value Percentages:
application_date                            0.000000
client_id_document_date                     1.022086
credit_product                              0.000000
declared_official_income                    0.000000
manager                                     0.000000
requested_loan_amount                       0.000000
city                                        0.000000
employer_bin_gcvp                          14.673402
delinquency_days_6m_credit_bureau           1.922776
completed_loans_count_credit_bureau         1.922776
total_outstanding_balance_credit_bureau     1.922776
active_loans_count_credit_bureau            1.922776
credit_bureau_inquiries_30d                 1.922776
loan_term_months                           14.254386
cenz_15_1_maturity_flag                     0.000000
fpd_15                                      0.000000
cenz_15_2_maturity_flag                     0.000000
spd_15                                      0.000000
active_delinquency_

In [12]:
### Duplicated Values
df.duplicated().sum()

15

In [13]:
df.sample(5,random_state=42)

,application_date,client_id_document_date,credit_product,declared_official_income,manager,requested_loan_amount,city,employer_bin_gcvp,delinquency_days_6m_credit_bureau,completed_loans_count_credit_bureau,total_outstanding_balance_credit_bureau,active_loans_count_credit_bureau,credit_bureau_inquiries_30d,loan_term_months,cenz_15_1_maturity_flag,fpd_15,cenz_15_2_maturity_flag,spd_15,active_delinquency_days
2281,2022-04-29,2021-08-17,Рефинансиров,250000.0,Егембердиева,1478908,Шымкент,100540004873/,0.0,0.0,478907.0,1,4.0,6.0,1,0,1,0,0
17521,2022-06-20,2013-01-03,Кредит налич,150000.0,Ќамбар,300000,Алматы,990540004827/,0.0,10.0,1510745.0,2,0.0,10.0,1,0,1,0,0
8779,2022-04-19,2009-02-17,Автокредитов,1500000.0,Налыбек,5600000,Караганд,NaN,0.0,3.0,4699601.0,4,1.0,NaN,1,0,1,0,0
7251,2022-06-27,2015-09-26,Ипотека,218764.0,Бугыбаева,6640000,Актобе,020540001554/,0.0,2.0,215616.0,1,2.0,6.0,1,0,1,0,0
12528,2022-04-11,2016-01-05,Кредит налич,0.0,Айтбайќызы,1200000,Кызылорд,150340008622/,0.0,3.0,148919.0,3,1.0,6.0,1,0,1,0,0


In [23]:
df['manager'].nunique()

908

In [26]:
# Check the distribution of a categorical column
print(df['credit_product'].value_counts())

credit_product
Кредит налич     7961
Автокредитов     7006
Ипотека          4654
#залоговыйкр     3213
Рефинансиров     1940
#картакарта_v     734
Товарный кре       28
Name: count, dtype: int64


In [29]:
print(df['city'].value_counts().head(5))

city
Алматы      5054
Нур-Султ    3808
Шымкент     2692
Караганд    2183
Усть-Кам    1340
Name: count, dtype: int64


In [36]:
pd.crosstab(df['city'],df['credit_product'],normalize='index').head(5) * 100

credit_product,#залоговыйкр,#картакарта_v,Автокредитов,Ипотека,Кредит налич,Рефинансиров,Товарный кре
city,,,,,,,
MOB_APP,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Актау,13.673806,0.494234,34.431631,11.696870,34.431631,5.271829,0.000000
Актобе,25.555556,0.606061,18.888889,18.080808,29.595960,7.272727,0.000000
Алматы,10.071231,10.645034,27.166601,15.433320,30.886427,5.619311,0.178077
Атырау,20.570012,0.743494,27.261462,22.924411,20.446097,7.682776,0.371747


In [39]:
pd.crosstab(df['credit_product'],df['fpd_15'],normalize='index') * 100

fpd_15,0,1
credit_product,,
#залоговыйкр,99.595394,0.404606
#картакарта_v,100.000000,0.000000
Автокредитов,99.857265,0.142735
Ипотека,99.871079,0.128921
Кредит налич,99.133275,0.866725
Рефинансиров,99.329897,0.670103
Товарный кре,100.000000,0.000000


In [47]:
print('First Payment Default (fpd_15) distribution - 1 means default, 0 means no default:', df['fpd_15'].value_counts())

First Payment Default (fpd_15) distribution - 1 means default, 0 means no default: fpd_15
0    25425
1      111
Name: count, dtype: int64
